### Decision Tree Model

In [170]:
""" 
This notebook implements a decision tree model
with target GMM weights , input : axle loading
"""

' \nThis notebook implements a decision tree model\nwith target GMM weights , input : axle loading\n'

In [171]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Capstone_Data") \
    .master("local[*]") \
    .getOrCreate()

In [172]:
#historical dataset 2019 - 2023
df = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_19_23.csv", header=True, inferSchema=True)

<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\rubie\AppData\Local\Temp\ipykernel_19596\3529866632.py:2: SyntaxWarning: invalid escape sequence '\R'
  df = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_19_23.csv", header=True, inferSchema=True)


In [173]:
df2 = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_24_25.csv", header=True, inferSchema=True)

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\rubie\AppData\Local\Temp\ipykernel_19596\3059976666.py:1: SyntaxWarning: invalid escape sequence '\R'
  df2 = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_24_25.csv", header=True, inferSchema=True)


In [174]:
#aggregating both datasets into 1
df3 = df.union(df2)

In [175]:
#Creating a date column
from pyspark.sql.functions import col, concat, lpad
df4 = df3.withColumn(
    "date",
    concat(
        col("year"),
        lpad(col("month"), 2, "0"),  
        lpad(col("day"), 2, "0")
    )
)


In [176]:
#Filtering for QueensBound Data and casting gvw column as float type
from pyspark.sql import functions as F

df5 = df4.filter("direction = 'QB'").withColumn("gvw", F.col("gvw").cast("float"))

In [177]:
df5 = df5.fillna(0)

In [178]:
# Removing outliers *asssume pandas df, targets gvw col
#Outliers are removed using vehicle gvw 
def remove_outliers_iqr(df, factor):
    Q1 = df["gvw"].quantile(0.25)
    Q3 = df["gvw"].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR

    outliers_iqr = df[(df["gvw"] < lower_bound) | (df["gvw"] > upper_bound)]
    
    print(f"Outlier Count: {len(outliers_iqr)}")
    print(f"Original Count: {len(df)}")
    print(f"Remaining count: {len(df) - len(outliers_iqr)}")
    print(f"Outlier percentage: {len(outliers_iqr)/len(df)*100:.2f}%")
    print(f"Upper bound: {upper_bound:.2f}, Lower bound: {lower_bound:.2f}")
    
    # Return filtered dataframe
    return df[(df["gvw"] >= lower_bound) & (df["gvw"] <= upper_bound)], upper_bound, lower_bound

In [179]:
#eliminate outliers from spark df
#Outliers are removed using vehicle gvw 
from pyspark.sql import functions as F

def remove_outliers_iqr2(df_spark, factor):
    # Step 1: Get Q1 and Q3
    q1, q3 = df_spark.approxQuantile("gvw", [0.25, 0.75], 0.01)

    # Step 2: Compute IQR
    iqr = q3 - q1

    # Step 3: Define bounds
    lower_bound = q1 - factor * iqr
    upper_bound = q3 + factor * iqr

    print(f"Upper bound: {upper_bound}, Lower bound: {lower_bound}")

    # Step 4: Filter outliers
    return df_spark.filter((F.col("gvw") >= lower_bound) & (F.col("gvw") <= upper_bound)) , upper_bound, lower_bound

In [180]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import GaussianMixture as spark_GM

def fit_gmm_spark(df_spark, cols, num_clusters, max_iter):
    
    assembler = VectorAssembler(inputCols=cols, outputCol="features")
    df_input = assembler.transform(df_spark)

    gmm = spark_GM(k=num_clusters, maxIter=max_iter)

    # Fit the model
    model = gmm.fit(df_input)

    #weights for each record
    df_spark2 = model.transform(df_input)

    return model, df_spark2


In [181]:
#format the "probability" col for df_spark2
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def format_dt_inputs(df_spark2):
    # Extract probability vector into individual cluster columns
    df_spark3 = (df_spark2
        .withColumn("probability_array", vector_to_array(col("probability")))
        .withColumn("cluster1", col("probability_array")[0])
        .withColumn("cluster2", col("probability_array")[1])
        .withColumn("cluster3", col("probability_array")[2])
        .drop("probability_array", "probability")  # Can drop multiple columns at once
    )

    # Parse date and extract temporal features
    df_spark3 = (df_spark3
        .withColumn("parsed_date", F.to_date(F.col("date").cast(StringType()), "yyMMdd"))
        .withColumn("week_of_year", F.weekofyear("parsed_date"))
        .withColumn("day_of_week", F.dayofweek("parsed_date") - 1)  # Convert to 0-6 range (Monday=0)
        .drop("parsed_date")
    )

    return df_spark3

In [182]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

#Excepts a pandas df/samples
def dt_grid_search(X_train, y_train, X_test, y_test, model, param_grid, cv=5):    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',  # Use negative MSE for scikit-learn convention
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Calculate additional metrics
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("="*50)
    print("REGRESSION GRID SEARCH RESULTS")
    print("="*50)
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV Score (-MSE): {grid_search.best_score_}")
    print(f"Test MSE: {mse}")
    
    return grid_search

In [183]:
def predict_cluster_weights(input_data, models):
    k1_wt = models[0].predict(input_data)[0]
    k2_wt = models[1].predict(input_data)[0]
    k3_wt = models[2].predict(input_data)[0]

    return [k1_wt, k2_wt, k3_wt]


Class 2

In [184]:
import pandas as pd
#filtering by class
class_num = 2
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 6664.0, Lower bound: 196.0


In [185]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [186]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.9960 (99.60%)
Component 1: 0.0002 (0.02%)
Component 2: 0.0038 (0.38%)


In [187]:
df_spark3 = format_dt_inputs(df_spark2)

In [188]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [189]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 438634
Test set size: 187986


In [190]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [191]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(4386)}
Best CV Score (-MSE): -0.0037269377781308315
Test MSE: 0.0037882698739469403


In [192]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster1.pdf'

In [193]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(39477)}
Best CV Score (-MSE): -4.185169945421616e-05
Test MSE: 3.8217309398253596e-05


In [194]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster2.pdf'

In [195]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(4386)}
Best CV Score (-MSE): -0.0034611479344813455
Test MSE: 0.0035310825616933555


In [196]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster3.pdf'

In [197]:
#No error handling implemented yet, care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

#arg vars are above this pd df is for organizing for model passing
input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.989 , 98.892%
Cluster 2 predicted weight: 0.000 , 0.020%
Cluster 3 predicted weight: 0.011 , 1.079%


Class 3

In [198]:
import pandas as pd
#filtering by class
class_num = 3
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 2.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 12790.0, Lower bound: -1970.0


In [199]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [200]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.0124 (1.24%)
Component 1: 0.9751 (97.51%)
Component 2: 0.0124 (1.24%)


In [201]:
df_spark3 = format_dt_inputs(df_spark2)

In [202]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [203]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 70242
Test set size: 30104


In [204]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [205]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2809)}
Best CV Score (-MSE): -0.0026514168204001425
Test MSE: 0.002747268206392141


In [206]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster1.pdf'

In [207]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(40), 'min_samples_leaf': np.int64(2809)}
Best CV Score (-MSE): -0.010605667281600567
Test MSE: 0.01098907282556856


In [208]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster2.pdf'

In [209]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2809)}
Best CV Score (-MSE): -0.0026514168204001425
Test MSE: 0.002747268206392141


In [210]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster3.pdf'

In [211]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.015 , 1.471%
Cluster 2 predicted weight: 0.971 , 97.057%
Cluster 3 predicted weight: 0.015 , 1.471%


Class 4

In [212]:
import pandas as pd
#filtering by class
class_num = 4
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.3
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 63208.0, Lower bound: -9368.0


In [213]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [214]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [215]:
df_spark3 = format_dt_inputs(df_spark2)

In [216]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [217]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 296580
Test set size: 127107


In [218]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [219]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2965)}
Best CV Score (-MSE): -1.2126992500673904e-26
Test MSE: 1.1362516078137847e-24


In [220]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster1.pdf'

In [221]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2965)}
Best CV Score (-MSE): -5.82366263136447e-27
Test MSE: 8.30839993441082e-25


In [222]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster2.pdf'

In [223]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2965)}
Best CV Score (-MSE): -1.449874753311162e-27
Test MSE: 1.0043715448675874e-24


In [224]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster3.pdf'

In [225]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 5

In [226]:
import pandas as pd
#filtering by class
class_num = 5
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 44052.0, Lower bound: -9162.0


In [227]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [228]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [229]:
df_spark3 = format_dt_inputs(df_spark2)

In [230]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [231]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 25314
Test set size: 10849


In [232]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [233]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(20), 'min_samples_leaf': np.int64(506)}
Best CV Score (-MSE): -1.53289578584347e-17
Test MSE: 1.5138766202987642e-17


In [234]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster1.pdf'

In [235]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(1518)}
Best CV Score (-MSE): -6.044019356417964e-17
Test MSE: 5.973909090640599e-17


In [236]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster2.pdf'

In [237]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(60), 'min_samples_leaf': np.int64(2025)}
Best CV Score (-MSE): -1.4902244345103918e-17
Test MSE: 1.4693825120531453e-17


In [238]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster3.pdf'

In [239]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 6

In [240]:
import pandas as pd
#filtering by class
class_num = 6
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 74704.0, Lower bound: -2534.0


In [241]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [242]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [243]:
df_spark3 = format_dt_inputs(df_spark2)

In [244]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [245]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 7639
Test set size: 3274


In [246]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [247]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(76)}
Best CV Score (-MSE): -1.3792418989191413e-19
Test MSE: 1.3948483270122106e-19


In [248]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster1.pdf'

In [249]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(76)}
Best CV Score (-MSE): -4.265863963515099e-22
Test MSE: 4.323352510777199e-22


In [250]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster2.pdf'

In [251]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(70), 'min_samples_leaf': np.int64(76)}
Best CV Score (-MSE): -1.5356270035665923e-19
Test MSE: 1.5551679104296442e-19


In [252]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster3.pdf'

In [253]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 7

In [254]:
import pandas as pd
#filtering by class
class_num = 7
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 8)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 115145.0, Lower bound: -1535.0


In [255]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [256]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [257]:
df_spark3 = format_dt_inputs(df_spark2)

In [258]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [259]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 68463
Test set size: 29342


In [260]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [261]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(3423)}
Best CV Score (-MSE): -1.3283441444542412e-27
Test MSE: 3.7972356457883606e-27


In [262]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster1.pdf'

In [263]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(80), 'min_samples_leaf': np.int64(2738)}
Best CV Score (-MSE): -2.653518908937039e-27
Test MSE: 6.583744808953402e-28


In [264]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster2.pdf'

In [265]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(2738)}
Best CV Score (-MSE): -1.202895784522931e-27
Test MSE: 5.467828691546373e-27


In [266]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster3.pdf'

In [267]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 8

In [268]:
import pandas as pd
#filtering by class
class_num = 8
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 5)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 76745.0, Lower bound: 5985.0


In [269]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [270]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [271]:
df_spark3 = format_dt_inputs(df_spark2)

In [272]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [273]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 383754
Test set size: 164467


In [274]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [275]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(50), 'min_samples_leaf': np.int64(3837)}
Best CV Score (-MSE): -4.377549665745242e-14
Test MSE: 4.3928916044135406e-14


In [276]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster1.pdf'

In [277]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(80), 'min_samples_leaf': np.int64(3837)}
Best CV Score (-MSE): -2.601457810380763e-14
Test MSE: 2.6477057828893716e-14


In [278]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster2.pdf'

In [279]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(90), 'min_samples_leaf': np.int64(3837)}
Best CV Score (-MSE): -1.3670775864134527e-13
Test MSE: 1.3704787616850426e-13


In [280]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster3.pdf'

In [281]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 9

In [282]:
import pandas as pd
#filtering by class
class_num = 9
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 121400.0, Lower bound: -1800.0


In [283]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [284]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [285]:
df_spark3 = format_dt_inputs(df_spark2)

In [286]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [287]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 15939
Test set size: 6832


In [288]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [289]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(159)}
Best CV Score (-MSE): -2.2482066457225994e-27
Test MSE: 5.150514319473922e-28


In [290]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster1.pdf'

In [291]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(159)}
Best CV Score (-MSE): -2.24511879645516e-27
Test MSE: 5.165403710936228e-28


In [292]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster2.pdf'

In [293]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(159)}
Best CV Score (-MSE): -2.2446077512249698e-27
Test MSE: 5.1678651847163915e-28


In [294]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster3.pdf'

In [295]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 10

In [296]:
import pandas as pd
#filtering by class
class_num = 10
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 18)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 175600.0, Lower bound: 3280.0


In [297]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [298]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [299]:
df_spark3 = format_dt_inputs(df_spark2)

In [300]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [301]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 65145
Test set size: 27920


In [302]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [303]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(50), 'min_samples_leaf': np.int64(6514)}
Best CV Score (-MSE): -1.1966919253235607e-27
Test MSE: 2.218810520140538e-26


In [304]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster1.pdf'

In [305]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(90), 'min_samples_leaf': np.int64(6514)}
Best CV Score (-MSE): -1.3900059963059144e-27
Test MSE: 2.218810520140538e-26


In [306]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster2.pdf'

In [307]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(80), 'min_samples_leaf': np.int64(6514)}
Best CV Score (-MSE): -1.060513835719802e-27
Test MSE: 3.745673777160437e-27


In [308]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster3.pdf'

In [309]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 11

In [310]:
import pandas as pd
#filtering by class
class_num = 11
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.7
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 109110.0, Lower bound: 20670.0


In [311]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [312]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [313]:
df_spark3 = format_dt_inputs(df_spark2)

In [314]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [315]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 48510
Test set size: 20791


In [316]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [317]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1174212812472893e-26
Test MSE: 5.648303658618644e-26


In [318]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster1.pdf'

In [319]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1179271883269167e-26
Test MSE: 5.649160357625781e-26


In [320]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster2.pdf'

In [321]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1187932813494276e-26
Test MSE: 5.650586694787892e-26


In [322]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster3.pdf'

In [323]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 12

In [324]:
import pandas as pd
#filtering by class
class_num = 12
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 7)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 2.0
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 100380.0, Lower bound: 19980.0


In [325]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [326]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [327]:
df_spark3 = format_dt_inputs(df_spark2)

In [328]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [329]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 22626
Test set size: 9697


In [330]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [331]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(905)}
Best CV Score (-MSE): -3.536125114572694e-28
Test MSE: 5.917122834799241e-28


In [332]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster1.pdf'

In [333]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(905)}
Best CV Score (-MSE): -3.9141096860311983e-28
Test MSE: 7.558818606382782e-28


In [334]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster2.pdf'

In [335]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(50), 'min_samples_leaf': np.int64(1131)}
Best CV Score (-MSE): -3.5264799389448097e-28
Test MSE: 7.558815670119165e-28


In [336]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=2)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster3.pdf'

In [337]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%
